# База данных SmartVR

База данных предназначена для хранения конфигураций мобильной VR-системы,
подключённых устройств, результатов калибровки, VR-сессий,
телеметрии и событий органов управления.

В качестве СУБД используется SQLite.

In [ ]:
%pip install -U jupysql sqlalchemy
%load_ext sql
%sql sqlite:///SmartVR.db

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS Device (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    hardware_uid TEXT NOT NULL UNIQUE,
    name TEXT NOT NULL,

    type TEXT NOT NULL CHECK (
        type IN ('SMARTPHONE', 'CONTROLLER', 'RECEIVER')
    ),

    model TEXT,
    firmware_version TEXT,

    created_at INTEGER NOT NULL,
    last_seen_at INTEGER
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS VRSetup (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    name TEXT NOT NULL,
    description TEXT,

    created_at INTEGER NOT NULL,
    updated_at INTEGER NOT NULL
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS SetupDevice (
    setup_id INTEGER NOT NULL,
    device_id INTEGER NOT NULL,

    role TEXT NOT NULL CHECK (
        role IN (
            'HMD',
            'LEFT_CONTROLLER',
            'RIGHT_CONTROLLER',
            'TRACKER',
            'RECEIVER'
        )
    ),

    enabled INTEGER NOT NULL DEFAULT 1 CHECK (
        enabled IN (0, 1)
    ),

    PRIMARY KEY (setup_id, device_id),

    FOREIGN KEY (setup_id)
        REFERENCES VRSetup(id)
        ON DELETE CASCADE,

    FOREIGN KEY (device_id)
        REFERENCES Device(id)
        ON DELETE CASCADE
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS RadioPairing (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    controller_id INTEGER NOT NULL,
    receiver_id INTEGER NOT NULL,

    protocol TEXT NOT NULL CHECK (
        protocol IN (
            'ESP_NOW',
            'NRF24',
            'CUSTOM_2_4_GHZ'
        )
    ),

    radio_address TEXT,
    channel INTEGER,
    paired_at INTEGER NOT NULL,

    FOREIGN KEY (controller_id)
        REFERENCES Device(id)
        ON DELETE CASCADE,

    FOREIGN KEY (receiver_id)
        REFERENCES Device(id)
        ON DELETE CASCADE,

    CHECK (controller_id <> receiver_id)
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS Calibration (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    setup_id INTEGER NOT NULL,
    device_id INTEGER NOT NULL,

    gyro_bias_x REAL NOT NULL,
    gyro_bias_y REAL NOT NULL,
    gyro_bias_z REAL NOT NULL,

    accel_bias_x REAL NOT NULL,
    accel_bias_y REAL NOT NULL,
    accel_bias_z REAL NOT NULL,

    orientation_offset_x REAL NOT NULL,
    orientation_offset_y REAL NOT NULL,
    orientation_offset_z REAL NOT NULL,
    orientation_offset_w REAL NOT NULL,

    position_offset_x REAL,
    position_offset_y REAL,
    position_offset_z REAL,

    created_at INTEGER NOT NULL,

    FOREIGN KEY (setup_id, device_id)
        REFERENCES SetupDevice(setup_id, device_id)
        ON DELETE CASCADE
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS Session (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    setup_id INTEGER NOT NULL,

    started_at INTEGER NOT NULL,
    ended_at INTEGER,

    status TEXT NOT NULL CHECK (
        status IN ('ACTIVE', 'FINISHED', 'ABORTED')
    ),

    FOREIGN KEY (setup_id)
        REFERENCES VRSetup(id)
        ON DELETE CASCADE
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS Telemetry (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    session_id INTEGER NOT NULL,
    device_id INTEGER NOT NULL,

    timestamp_us INTEGER NOT NULL,
    sequence_no INTEGER NOT NULL,

    accel_x REAL NOT NULL,
    accel_y REAL NOT NULL,
    accel_z REAL NOT NULL,

    gyro_x REAL NOT NULL,
    gyro_y REAL NOT NULL,
    gyro_z REAL NOT NULL,

    quat_x REAL NOT NULL,
    quat_y REAL NOT NULL,
    quat_z REAL NOT NULL,
    quat_w REAL NOT NULL,

    position_x REAL,
    position_y REAL,
    position_z REAL,

    tracking_quality REAL CHECK (
        tracking_quality IS NULL
        OR tracking_quality BETWEEN 0.0 AND 1.0
    ),

    FOREIGN KEY (session_id)
        REFERENCES Session(id)
        ON DELETE CASCADE,

    FOREIGN KEY (device_id)
        REFERENCES Device(id)
);

In [ ]:
%%sql

CREATE TABLE IF NOT EXISTS InputEvent (
    id INTEGER PRIMARY KEY AUTOINCREMENT,

    session_id INTEGER NOT NULL,
    device_id INTEGER NOT NULL,

    timestamp_us INTEGER NOT NULL,

    control_code TEXT NOT NULL,

    event_type TEXT NOT NULL CHECK (
        event_type IN ('PRESS', 'RELEASE', 'ANALOG')
    ),

    value REAL,

    FOREIGN KEY (session_id)
        REFERENCES Session(id)
        ON DELETE CASCADE,

    FOREIGN KEY (device_id)
        REFERENCES Device(id)
);

In [ ]:
%%sql

CREATE INDEX IF NOT EXISTS idx_setup_device_device
ON SetupDevice(device_id);

CREATE INDEX IF NOT EXISTS idx_calibration_setup_device
ON Calibration(setup_id, device_id, created_at);

CREATE INDEX IF NOT EXISTS idx_session_setup
ON Session(setup_id);

CREATE INDEX IF NOT EXISTS idx_telemetry_session_time
ON Telemetry(session_id, timestamp_us);

CREATE INDEX IF NOT EXISTS idx_telemetry_device_time
ON Telemetry(device_id, timestamp_us);

CREATE INDEX IF NOT EXISTS idx_input_event_session_time
ON InputEvent(session_id, timestamp_us);

CREATE INDEX IF NOT EXISTS idx_radio_pairing_receiver
ON RadioPairing(receiver_id);

In [ ]:
%%sql

SELECT name
FROM sqlite_master
WHERE type = 'table'
  AND name NOT LIKE 'sqlite_%'
ORDER BY name;

In [ ]:
%sql PRAGMA foreign_key_list(SetupDevice);

Заполнение тестовыми данными

In [ ]:

import sqlite3
import time
from pathlib import Path

path = Path("SmartVR.db")
if not path.exists():
    raise FileNotFoundError(f"Не найдена база: {path.resolve()}")

with sqlite3.connect(path) as db:
    db.execute("PRAGMA foreign_keys = ON")

    if db.execute("SELECT 1 FROM VRSetup WHERE name = '[DEMO] SmartVR'").fetchone():
        print("Демо-данные уже существуют.")
    else:
        now = int(time.time())
        ids = {}

        devices = [
            ("SMARTVR-DEMO-HMD", "Демо: смартфон HMD", "SMARTPHONE"),
            ("SMARTVR-DEMO-L", "Демо: левый контроллер", "CONTROLLER"),
            ("SMARTVR-DEMO-R", "Демо: правый контроллер", "CONTROLLER"),
            ("SMARTVR-DEMO-RX", "Демо: ESP32-C3 приёмник", "RECEIVER"),
        ]

        for uid, name, kind in devices:
            cur = db.execute(
                "INSERT INTO Device (hardware_uid, name, type, created_at) VALUES (?, ?, ?, ?)",
                (uid, name, kind, now)
            )
            ids[uid] = cur.lastrowid

        setupId = db.execute(
            "INSERT INTO VRSetup (name, description, created_at, updated_at) VALUES (?, ?, ?, ?)",
            ("[DEMO] SmartVR", "Тестовая конфигурация для ТК", now, now)
        ).lastrowid

        roles = [
            ("SMARTVR-DEMO-HMD", "HMD"),
            ("SMARTVR-DEMO-L", "LEFT_CONTROLLER"),
            ("SMARTVR-DEMO-R", "RIGHT_CONTROLLER"),
            ("SMARTVR-DEMO-RX", "RECEIVER"),
        ]

        db.executemany(
            "INSERT INTO SetupDevice (setup_id, device_id, role) VALUES (?, ?, ?)",
            [(setupId, ids[uid], role) for uid, role in roles]
        )

        for uid in ("SMARTVR-DEMO-L", "SMARTVR-DEMO-R"):
            deviceId = ids[uid]

            db.execute(
                """INSERT INTO RadioPairing
                   (controller_id, receiver_id, protocol, paired_at)
                   VALUES (?, ?, 'ESP_NOW', ?)""",
                (deviceId, ids["SMARTVR-DEMO-RX"], now)
            )

            db.execute(
                """INSERT INTO Calibration (
                    setup_id, device_id,
                    gyro_bias_x, gyro_bias_y, gyro_bias_z,
                    accel_bias_x, accel_bias_y, accel_bias_z,
                    orientation_offset_x, orientation_offset_y,
                    orientation_offset_z, orientation_offset_w,
                    created_at
                ) VALUES (?, ?, 0,0,0, 0,0,0, 0,0,0,1, ?)""",
                (setupId, deviceId, now)
            )

        sessionId = db.execute(
            """INSERT INTO Session
               (setup_id, started_at, ended_at, status)
               VALUES (?, ?, ?, 'FINISHED')""",
            (setupId, now - 120, now)
        ).lastrowid

        for i, uid in enumerate(("SMARTVR-DEMO-L", "SMARTVR-DEMO-R"), 1):
            db.execute(
                """INSERT INTO Telemetry (
                    session_id, device_id, timestamp_us, sequence_no,
                    accel_x, accel_y, accel_z,
                    gyro_x, gyro_y, gyro_z,
                    quat_x, quat_y, quat_z, quat_w,
                    tracking_quality
                ) VALUES (?, ?, ?, ?, 0,0,9.81, 0,0,0, 0,0,0,1, 1.0)""",
                (sessionId, ids[uid], (now - 100) * 1000000, i)
            )

        for eventType, value, offset in [
            ("PRESS", 1.0, 90),
            ("RELEASE", 0.0, 89),
        ]:
            db.execute(
                """INSERT INTO InputEvent
                   (session_id, device_id, timestamp_us, control_code, event_type, value)
                   VALUES (?, ?, ?, 'A', ?, ?)""",
                (sessionId, ids["SMARTVR-DEMO-L"], (now - offset) * 1000000,
                 eventType, value)
            )

        print("Демо-данные успешно добавлены.")


Демо-данные успешно добавлены.


Очищение тестовых данных

In [ ]:

import sqlite3

with sqlite3.connect("SmartVR.db") as db:
    db.execute("PRAGMA foreign_keys = ON")
    db.execute("DELETE FROM VRSetup WHERE name = '[DEMO] SmartVR'")
    db.execute("DELETE FROM Device WHERE hardware_uid LIKE 'SMARTVR-DEMO-%'")

print("Демо-данные удалены.")
